In [ ]:
%conda install geopandas

In [ ]:
%conda install plotly

In [ ]:
"""Load Boundaries"""

In [ ]:
from pathlib import Path
import pathlib
import os
from shapely.geometry import mapping
import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd

def prepare_boundary(dir_path):
    print("Running")
    directory = Path(dir_path)

    files = []
    for subdir in directory.iterdir():
            if subdir.is_dir():
                print("\nProcessing directory:", subdir.name)
                sub_directory = Path(subdir)
                for filename in sub_directory.glob("*.shp"):
                    if filename.is_file():
                        print(filename)
                        current = gpd.read_file(filename)


                        # print("Attributes available:", list(gdf.columns))
                        print("Attributes available:", list(current.columns))


                        # print(current.head())
                        # print(current.columns)
                        # boundary = boundary.set_crs("EPSG:4326")
                        # boundary = boundary.to_crs("EPSG:4326")
                        # current.plot()
                        files.append(current)
    return files



# Process shape file
print("\n\nDenver")
print(os.listdir("."))
census_boundaries = prepare_boundary("colorado_census/")

print(type(census_boundaries))
county = (census_boundaries[1])
print(county.head())
print(county.columns)


# /Users/klinarez/Documents/em-public-dashboard/viz_scripts/Colorado

In [ ]:
import plotly.express as px
import geopandas as gpd
import json

# Your census boundaries list
census_boundaries = prepare_boundary("colorado_counties_only/")

# Combine all boundaries into one GeoDataFrame (if needed)
# Or loop through them individually
choropleths = []
county_index_map = {}

for idx, gdf in enumerate(census_boundaries):
    # Convert to WGS84 if needed
    if gdf.crs is not None and gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    # Convert to GeoJSON
    geojson_data = json.loads(gdf.to_json())

    # Pick a column to color by (any numeric or categorical)
    # If none exists, create a dummy one
    if "CENSUS_ID" in gdf.columns:
        color_col = "CENSUS_ID"
    else:
        color_col = "idx"
        gdf[color_col] = idx

    
    # Build county:index dictionary entries
    for _, row in gdf.iterrows():
        county_name = row["COUNTY"]
        county_index = row[idx]
        county_index_map[county_name] = county_index


    fig = px.choropleth(
        gdf,
        geojson=geojson_data,
        locations=gdf.index,
        color=color_col,
        hover_data=[col for col in gdf.columns if col == "COUNTY"],
        projection="mercator"
    )

    choropleths.append(fig)


print(county_index_map)# Merge all choropleths into one final figure
final_fig = choropleths[0]
for fig in choropleths[1:]:
    for trace in fig.data:
        final_fig.add_trace(trace)

final_fig.update_geos(
    fitbounds="locations",
    visible=False
)

final_fig.update_layout(
    height=1000,
    margin=dict(l=0, r=0, b=0, t=0),
)

final_fig.show()
final_fig.write_html("colorado_census_boundaries_choropleth.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import geopandas as gpd
import shapely.geometry
from shapely.geometry import Polygon, MultiPolygon
import numpy as np
import os


census_boundaries = prepare_boundary("colorado_census/")

def extract_coordinates(shapefile, tolerance=0.0005):
    lons = []
    lats = []

    # Extract border geometry
    for geom in shapefile.geometry.simplify(tolerance, preserve_topology=True):
        # If it's a single polygon
        if isinstance(geom, Polygon):
            polys = [geom]

        # If it's a multipolygon
        elif isinstance(geom, MultiPolygon):
            polys = list(geom.geoms)

        else:
            continue

        for poly in polys:
            if poly.is_empty:
                continue

        # Extract exterior coordinates for each polygon
        for poly in polys:
            x, y = poly.exterior.xy
            lons.extend(x)
            lats.extend(y)

            lons.append(None)
            lats.append(None)

        # Extract county name

    return lons, lats, 



fig = go.Figure()

# Census Colors
colors = ["red", "blue", "green"]


# List of traces for each census boundary
for idx, elem in enumerate(census_boundaries):
    print(elem.columns)
    # All .dbf attributes EXCEPT geometry
    attr_cols = [col for col in elem.columns if col != "geometry"]
    # Convert the attributes into customdata
    custom_data = elem[attr_cols].values
    # Build a hovertemplate dynamically
    hover_lines = []
    for i, col in enumerate(attr_cols):
        hover_lines.append(f"{col}: %{{customdata[{i}]}}")
    hovertemplate = "<br>".join(hover_lines) + "<extra></extra>"



    name = f"Census Boundary: {idx+1}"
    lons, lats = extract_coordinates(elem.geometry)
    fig.add_trace(go.Scattermap(
        lon=lons,
        lat=lats,
        mode="lines",
        line=dict(color=colors[idx], width=2),
        name=name,
        # .DBF ATTRIBUTES
        customdata=custom_data,
        hovertemplate=hovertemplate
    ))


center_lon, center_lat = -105.5844, 39.7392

fig.update_layout(legend=dict(
    yanchor="top",
    y=0.75,
    xanchor="right",
))

fig.update_layout(
    height=1000,
    margin=dict(l=0, r=0, b=0, t=0),
    autosize=True,
    map=dict(
        center=dict(lat=center_lat, lon=center_lon),
        zoom=6,
        style="open-street-map"
    )
)

fig.show()
fig.write_html("colorado_census_boundaries.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
# choropleth county indices

In [ ]:
global_counter = 0
county_index_map = {}

choropleths = []

census_boundaries = prepare_boundary("colorado_counties_only/")


for gdf in census_boundaries:
    if gdf.crs is not None and gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    gdf["county_index"] = range(global_counter, global_counter + len(gdf))
    global_counter += len(gdf)

    county_col = "COUNTY" if "COUNTY" in gdf.columns else None
    if county_col:
        county_index_map.update(dict(zip(gdf[county_col], gdf["county_index"])))

    color_col = "CENSUS_ID" if "CENSUS_ID" in gdf.columns else "county_index"

    geojson_data = json.loads(gdf.to_json())

    fig = px.choropleth(
        gdf,
        geojson=geojson_data,
        locations=gdf.index,
        featureidkey="id",
        color=color_col,
        hover_data=[county_col, "county_index"] if county_col else ["county_index"],
        projection="mercator",
    )

    choropleths.append(fig)

# Merge figs
final_fig = choropleths[0]
for fig in choropleths[1:]:
    for trace in fig.data:
        final_fig.add_trace(trace)

final_fig.update_geos(fitbounds="locations", visible=False)
final_fig.update_layout(height=1000, margin=dict(l=0, r=0, b=0, t=0))
final_fig.show()
final_fig.write_html("colorado_census_boundaries_choropleth.html",
                     include_plotlyjs="cdn", full_html=True)

In [ ]:
# scattermap coiunty idnices

import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon

# ----- Load and prep -----
census_boundaries = prepare_boundary("colorado_counties_only/")  # list[GeoDataFrame]

def extract_coordinates_for_geometry(geom, tolerance=0.0005):
    """
    Returns (lons, lats) arrays for exterior rings of Polygon/MultiPolygon.
    Inserts None between rings so Plotly breaks the line segments.
    """
    lons, lats = [], []
    if geom is None:
        return lons, lats

    try:
        geom_simpl = geom.simplify(tolerance, preserve_topology=True)
    except Exception:
        geom_simpl = geom  # fall back if simplify fails

    # Normalize to polygons list
    if isinstance(geom_simpl, Polygon):
        polys = [geom_simpl]
    elif isinstance(geom_simpl, MultiPolygon):
        polys = list(geom_simpl.geoms)
    else:
        # Try to iterate sub-geometries; keep only polygons
        polys = []
        try:
            for g in geom_simpl.geoms:
                if isinstance(g, Polygon):
                    polys.append(g)
        except Exception:
            pass

    for poly in polys:
        if poly.is_empty or poly.exterior is None:
            continue
        x, y = poly.exterior.xy
        lons.extend(list(x))
        lats.extend(list(y))
        # Separator
        lons.append(None)
        lats.append(None)

    return lons, lats

# Grape from array of shape files
all_gdf = census_boundaries[0]

# Identify county name column
county_col = "COUNTY" if "COUNTY" in all_gdf.columns else None
if county_col is None:
    # Try common alternatives; otherwise, pick first non-geometry string column
    candidates = [c for c in all_gdf.columns if c != "geometry"]
    string_cols = [c for c in candidates if all_gdf[c].astype(str).notna().all()]
    county_col = string_cols[0] if string_cols else None
if county_col is None:
    raise ValueError("Could not find a county name column (e.g., 'COUNTY'). Please provide one.")

# Create a unique county index across all rows
all_gdf["county_index"] = range(len(all_gdf))

# (Optional) keep a dict for programmatic use
county_index_map = dict(zip(all_gdf[county_col], all_gdf["county_index"]))
print(county_index_map)

# ----- Figure -----
fig = go.Figure()

# One trace per county for clean hover
for row in all_gdf.itertuples(index=False):
    county_name = getattr(row, county_col)
    idx_val = row.county_index
    lons, lats = extract_coordinates_for_geometry(row.geometry)

    if not lons:  # skip empty
        continue

    # Repeat county hover text for all vertices
    text = [f"{county_name} — index: {idx_val}"] * len(lons)

    fig.add_trace(go.Scattermap(
        lon=lons,
        lat=lats,
        mode="lines",
        line=dict(color="rgba(31,119,180,0.85)", width=2),
        name=county_name,  # legend entry per county (you can change to a group label)
        text=text,
        fill="toself",
        hovertemplate="%{text}<extra></extra>",
    ))

# Map center over Colorado (adjust if needed)
center_lon, center_lat = -105.5844, 39.7392

fig.update_layout(
    legend=dict(yanchor="top", y=0.75, xanchor="right"),
    height=1000,
    margin=dict(l=0, r=0, b=0, t=0),
    autosize=True,
    # MapLibre / scattermap configuration uses `layout.map`
    map=dict(
        center=dict(lat=center_lat, lon=center_lon),
        zoom=6,
        style="open-street-map"
    )
)

fig.show()
fig.write_html(
    "colorado_census_boundaries_scattermap.html",
    include_plotlyjs="cdn",
    full_html=True
)

In [ ]:
# LOAD POINT DATE

import pandas as pd

import emission.core.get_database as edb
import emission.core.wrapper.entry as ecwe
import emission.storage.decorations.analysis_timeseries_queries as esda
import emission.storage.decorations.trip_queries as esdt
import emission.storage.decorations.timeline as esdl
import emission.storage.timeseries.abstract_timeseries as esta
import emission.storage.timeseries.timequery as estt
import scaffolding
from uuid import UUID

from shapely.geometry import shape
from shapely.geometry import MultiPoint
from shapely.geometry import Point

%matplotlib inline

# Split UUIDs by program
program_uuid_map = {}
for ue in edb.get_uuid_db().find():
    program = ue['user_email'].split("_")[0]
    if program in program_uuid_map.keys():
        program_uuid_map[program].append(str(ue['uuid']))
    else:
        print(f"Found new program {program}, creating new list")
        program_uuid_map[program] = []
        program_uuid_map[program].append(str(ue['uuid']))


uuid_program_list = []
for ue in edb.get_uuid_db().find():
    program = ue['user_email'].split("_")[0]
    uuid_program_list.append({"program": program, "opcode": ue["user_email"], "user_id_str": str(ue['uuid'])})


year = None
month = None
program = "default"
study_type = "study"
include_test_users = False
# labels = {}
labels = {'MODE': [{'value': 'walk', 'base_mode': 'WALKING'}, {'value': 'e-bike', 'base_mode': 'E_BIKE'}, {'value': 'bike', 'base_mode': 'BICYCLING'}, {'value': 'bikeshare', 'base_mode': 'BICYCLING'}, {'value': 'scootershare', 'base_mode': 'E_SCOOTER'}, {'value': 'drove_alone', 'base_mode': 'CAR', 'passengers': 1}, {'value': 'shared_ride', 'base_mode': 'CAR', 'passengers': 2}, {'value': 'hybrid_drove_alone', 'base_mode': 'PHEV_CAR', 'passengers': 1}, {'value': 'hybrid_shared_ride', 'base_mode': 'PHEV_CAR', 'passengers': 2}, {'value': 'e_car_drove_alone', 'base_mode': 'E_CAR', 'passengers': 1}, {'value': 'e_car_shared_ride', 'base_mode': 'E_CAR', 'passengers': 2}, {'value': 'taxi', 'base_mode': 'TAXI'}, {'value': 'bus', 'base_mode': 'BUS'}, {'value': 'train', 'base_mode': 'TRAIN'}, {'value': 'free_shuttle', 'base_mode': 'BUS'}, {'value': 'air', 'base_mode': 'AIR'}, {'value': 'not_a_trip', 'base_mode': 'UNKNOWN'}, {'value': 'other', 'base_mode': 'OTHER'}], 'PURPOSE': [{'value': 'home'}, {'value': 'work'}, {'value': 'at_work'}, {'value': 'school'}, {'value': 'transit_transfer'}, {'value': 'shopping'}, {'value': 'meal'}, {'value': 'pick_drop_person'}, {'value': 'pick_drop_item'}, {'value': 'personal_med'}, {'value': 'access_recreation'}, {'value': 'exercise'}, {'value': 'entertainment'}, {'value': 'religious'}, {'value': 'other'}], 'REPLACED_MODE': [{'value': 'no_travel'}, {'value': 'walk'}, {'value': 'bike'}, {'value': 'bikeshare'}, {'value': 'scootershare'}, {'value': 'drove_alone'}, {'value': 'shared_ride'}, {'value': 'hybrid_drove_alone'}, {'value': 'hybrid_shared_ride'}, {'value': 'e_car_drove_alone'}, {'value': 'e_car_shared_ride'}, {'value': 'taxi'}, {'value': 'bus'}, {'value': 'train'}, {'value': 'free_shuttle'}, {'value': 'other'}], 'translations': {'en': {'walk': 'Walk', 'e-bike': 'E-bike', 'bike': 'Regular Bike', 'bikeshare': 'Bikeshare', 'scootershare': 'Scooter share', 'drove_alone': 'Gas Car Drove Alone', 'shared_ride': 'Gas Car Shared Ride', 'hybrid_drove_alone': 'Hybrid Drove Alone', 'hybrid_shared_ride': 'Hybrid Shared Ride', 'e_car_drove_alone': 'E-Car Drove Alone', 'e_car_shared_ride': 'E-Car Shared Ride', 'taxi': 'Taxi / Uber / Lyft', 'bus': 'Bus', 'train': 'Train', 'free_shuttle': 'Free Shuttle', 'air': 'Air', 'not_a_trip': 'Not a trip', 'no_travel': 'No travel', 'home': 'Home', 'work': 'To Work', 'at_work': 'At Work', 'school': 'School', 'transit_transfer': 'Transit transfer', 'shopping': 'Shopping', 'meal': 'Meal', 'pick_drop_person': 'Pick-up / Drop off Person', 'pick_drop_item': 'Pick-up / Drop off Item', 'personal_med': 'Personal / Medical', 'access_recreation': 'Access Recreation', 'exercise': 'Recreation / Exercise', 'entertainment': 'Entertainment / Social', 'religious': 'Religious', 'other': 'Other'}, 'es': {'walk': 'Caminando', 'e-bike': 'e-bicicleta', 'bike': 'Bicicleta', 'bikeshare': 'Bicicleta compartida', 'scootershare': 'Motoneta compartida', 'drove_alone': 'Coche de Gas, Condujo solo', 'shared_ride': 'Coche de Gas, Condujo con otros', 'hybrid_drove_alone': 'Coche Híbrido, Condujo solo', 'hybrid_shared_ride': 'Coche Híbrido, Condujo con otros', 'e_car_drove_alone': 'e-coche, Condujo solo', 'e_car_shared_ride': 'e-coche, Condujo con otros', 'taxi': 'Taxi / Uber / Lyft', 'bus': 'Autobús', 'train': 'Tren', 'free_shuttle': 'Colectivo gratuito', 'air': 'Avión', 'not_a_trip': 'No es un viaje', 'no_travel': 'No viajar', 'home': 'Casa', 'work': 'Trabajo', 'at_work': 'En el trabajo', 'school': 'Escuela', 'transit_transfer': 'Transbordo', 'shopping': 'Compras', 'meal': 'Comida', 'pick_drop_person': 'Recoger / Entregar Individuo', 'pick_drop_item': 'Recoger / Entregar Objeto', 'personal_med': 'Personal / Médico', 'access_recreation': 'Acceder a Recreación', 'exercise': 'Recreación / Ejercicio', 'entertainment': 'Entretenimiento / Social', 'religious': 'Religioso', 'other': 'Otros'}, 'lo': {'walk': 'ດ້ວຍການຍ່າງ', 'e-bike': 'ວຍລົດຈັກໄຟຟ້າ', 'bike': 'ລົດຖີບ', 'bikeshare': 'ດ້ວຍລົດຖີບທີ່ໃຫ້ເຊົ່າ', 'scootershare': 'ດ້ວຍລົດຈັກສະກູດເຕີ້ໃຫ້ເຊົ່າ ', 'drove_alone': 'ເດີນທາງ ດ້ວຍລົດໃຫ່ຍ ເຊີ່ງເປັນລົດທີ່ຂັບເອງ', 'shared_ride': 'ເດີນທາງດ້ວຍລົດໃຫ່ຍ ຮ່ວມກັບລົດຄົນອຶ່ນ', 'hybrid_drove_alone': 'ລົດປະສົມຂັບຄົນດຽວ', 'hybrid_shared_ride': 'ລົດປະສົມທີ່ໃຊ້ຮ່ວມກັນ', 'e_car_drove_alone': 'ດ້ວຍການຂັບລົດໄຟຟ້າໄປເອງ', 'e_car_shared_ride': 'ດ້ວຍການຈ້າງລົດໄຟຟ້າໄປ', 'taxi': 'ແທັກຊີ', 'bus': 'ລົດເມ', 'train': 'ລົດໄຟ', 'free_shuttle': 'ລົດຮັບສົ່ງຟຣີ', 'air': 'ຍົນ', 'not_a_trip': 'ບໍ່ແມ່ນການເດີນທາງ', 'no_travel': 'ບໍ່ມີການເດີນທາງ', 'home': 'ບ້ານ', 'work': 'ໄປເຮັດວຽກ', 'at_work': 'ຢູ່ບ່ອນເຮັດວຽກ', 'school': 'ໄປໂຮງຮຽນ', 'transit_transfer': 'ການຖ່າຍໂອນການເດີນທາງ', 'shopping': 'ຊອບປິ້ງ', 'meal': 'ອາຫານ', 'pick_drop_person': 'ໄປຮັບ ຫລື ສົນ ຄົນ', 'pick_drop_item': 'ໄປຮັບ ຫລື ສົ່ງສິນຄ້າ', 'personal_med': 'ໄປຫາໝໍ', 'access_recreation': 'ເຂົ້າເຖິງການພັກຜ່ອນ', 'exercise': 'ພັກຜ່ອນ/ອອກກຳລັງກາຍ', 'entertainment': 'ບັນເທີງ/ສັງຄົມ', 'religious': 'ຈຸດປະສົງທາງສາດສະໜາ', 'other': 'ອື່ນໆ'}}}
use_imperial = True
sensed_algo_prefix = "cleaned"
survey_info = {}

label_units, short_label, label_units_lower, distance_col, weight_unit = scaffolding.get_units(use_imperial)
expanded_ct_sensed, file_suffix_sensed, quality_text_sensed, debug_df_sensed = await scaffolding.load_viz_notebook_sensor_inference_data(year,
                                                                            month,
                                                                            program,
                                                                            labels,
                                                                            include_test_users,
                                                                            sensed_algo_prefix)



import geopandas as gpd
from shapely.geometry import shape, MultiPoint

expanded_ct_sensed['start_end'] = None

expanded_ct_sensed["start_end"] = [
    MultiPoint([shape(start), shape(end)])
    for start, end in zip(
        expanded_ct_sensed["start_loc"],
        expanded_ct_sensed["end_loc"]
    )
]

# Now build the GeoDataFrame with the geometry column you actually created
expanded_ct_sensed_gdf = gpd.GeoDataFrame(expanded_ct_sensed, geometry="start_end")
expanded_ct_sensed_gdf.crs is None
expanded_ct_sensed_gdf = expanded_ct_sensed_gdf.set_crs(4269, allow_override=True)
expanded_ct_sensed_gdf.crs
# print(expanded_ct_sensed_gdf)
# Convert to geojson
expanded_ct_sensed_gdf.to_file("expanded_ct_sensed.geojson", driver="GeoJSON")
geojson_obj = gpd.read_file("expanded_ct_sensed.geojson")
print("GEOJSON saved")
print(geojson_obj)


import shapely as shp
import os

os.environ['SHAPE_RESTORE_SHX'] = 'YES'

print(expanded_ct_sensed_gdf.primary_mode.unique())
# Filter for bike trips
expanded_ct_sensed_gdf.rename(columns={"data_user_input_mode_confirm": "BICYCLING"}, inplace = True)
bike_trips = expanded_ct_sensed_gdf[expanded_ct_sensed_gdf.primary_mode == 'BICYCLING']
print(len(bike_trips), "bike trips")

denver_boundary = gpd.read_file("shape_files/denver_uza_akcicek.shp")

# Convert to gemoetries, get start and end points
bike_start_end_points = pd.concat([bike_trips.start_loc, bike_trips.end_loc]); 
bike_geo_start_end_points = gpd.GeoSeries(
    bike_start_end_points.apply(
        lambda d: Point(d["coordinates"])
    ),
    crs="EPSG:4269"
)
print((bike_start_end_points.iloc[0]))
bike_start_or_end_within = bike_geo_start_end_points[bike_geo_start_end_points.within(denver_boundary.geometry.iloc[0])]

non_bike_trips = expanded_ct_sensed_gdf.query('primary_mode == "IN_VEHICLE" | primary_mode == "WALKING" | primary_mode == "AIR_OR_HSR"')
print(len(non_bike_trips))
non_bike_start_end_points = pd.concat([non_bike_trips.start_loc, non_bike_trips.end_loc])

non_bike_geo_start_end_points = gpd.GeoSeries(non_bike_start_end_points.apply(lambda p: shp.geometry.Point(p["coordinates"])), crs="EPSG:4269")
non_bike_start_or_end_within = non_bike_geo_start_end_points[non_bike_geo_start_end_points.within(denver_boundary.geometry.iloc[0])]


# WHAT GETS LOADED IN
def prepare_boundary_points(filename):
    boundary = gpd.read_file(filename)
    boundary.plot()
    return boundary

# See if shape files have plotly equivalent - May not want to filter to specific cities. 
denver_boundary = prepare_boundary_points("shape_files/denver_uza_akcicek.shp")

In [ ]:
# Scattermap with hoverinfo  == toself

import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon

# ----- Load and prep -----
census_boundaries = prepare_boundary("colorado_counties_only/")  # list[GeoDataFrame]

# Ensure GeoDataFrame is in WGS84 (EPSG:4326)
def to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is not None and gdf.crs.to_string() != "EPSG:4326":
        return gdf.to_crs(epsg=4326)
    return gdf


#  Return (lons, lats) for exterior rings of Polygon/MultiPolygon.
def polygon_exterior_coords(geom, tolerance=0.0005):

    lons, lats = [], []
    if geom is None:
        return lons, lats

    try:
        geom_simpl = geom.simplify(tolerance, preserve_topology=True)
    except Exception:
        geom_simpl = geom

    # Normalize to list of polygons
    if isinstance(geom_simpl, Polygon):
        polys = [geom_simpl]
    elif isinstance(geom_simpl, MultiPolygon):
        polys = list(geom_simpl.geoms)
    else:
        polys = []
        try:
            for g in geom_simpl.geoms:
                if isinstance(g, Polygon):
                    polys.append(g)
        except Exception:
            pass

    for poly in polys:
        if poly.is_empty or poly.exterior is None:
            continue
        x, y = poly.exterior.xy
        lons.extend(list(x))
        lats.extend(list(y))
        # separator for multi-polygons in the same trace
        lons.append(None)
        lats.append(None)

    return lons, lats

from shapely.prepared import prep
prepared_counties = [
    (prep(row.geometry), row.county_index, row.geometry.bounds)
    for row in all_gdf.itertuples(index=False)
]


from shapely.geometry import Point

def find_county_for_point(lon, lat, prepared_counties):
    pt = Point(lon, lat)

    for geom_prepared, idx, bounds in prepared_counties:
        minx, miny, maxx, maxy = bounds

        # bounding-box prefilter
        if not (minx <= lon <= maxx and miny <= lat <= maxy):
            continue

        # precise containment test
        if geom_prepared.contains(pt):
            return idx



# Reproject and concatenate point type
reprojected = [to_wgs84(gdf) for gdf in census_boundaries]
all_gdf = gpd.GeoDataFrame(pd.concat(reprojected, ignore_index=True), crs="EPSG:4326")


##### COUNTY TRACES #####

# Identify county name column
county_col = "COUNTY" if "COUNTY" in all_gdf.columns else None
if county_col is None:
    candidates = [c for c in all_gdf.columns if c != "geometry"]
    string_cols = [c for c in candidates if all_gdf[c].astype(str).notna().all()]
    county_col = string_cols[0] if string_cols else None
if county_col is None:
    raise ValueError("Could not find a county name column (e.g., 'COUNTY'). Please provide one.")

# Create a unique county index across *all* features
all_gdf["county_index"] = range(len(all_gdf))

# Keep a dict if needed elsewhere
county_index_map = dict(zip(all_gdf[county_col], all_gdf["county_index"]))
index_to_county = dict(zip(all_gdf["county_index"], all_gdf[county_col]))
print(county_index_map)

# ----- Figure -----
fig = go.Figure()

# 1) Filled polygon per county (visuals)
for row in all_gdf.itertuples(index=False):
    county_name = getattr(row, county_col)
    idx_val = row.county_index

    lons, lats = polygon_exterior_coords(row.geometry)
    if not lons:
        continue

    fig.add_trace(go.Scattermap(
        lon=lons,
        lat=lats,
        mode="lines",
        fill="toself",                               # fill draws interior nicely
        fillcolor="rgba(0, 123, 255, 0.25)",         # semi-transparent fill
        line=dict(color="rgba(0, 123, 255, 0.85)", width=2),
        name=county_name,
        # Hover shows when crossing the drawn coordinates; interior hover
        # needs a marker layer (added below).
        hoverinfo="skip",                            # avoid edge-only hovers
    ))

# 2) Interior hover via center markers (one per county)
rep_pts = all_gdf.geometry.representative_point()   # guaranteed inside polygon
points_lon = [p.x for p in rep_pts]
points_lat = [p.y for p in rep_pts]

customdata = pd.DataFrame({
    "county_name": all_gdf[county_col],
    "county_index": all_gdf["county_index"]
}).values

fig.add_trace(go.Scattermap(
    lon=points_lon,
    lat=points_lat,
    mode="markers",
    marker=dict(size=10, color="rgba(0,0,0,1)"),
    name="County centers",
    customdata=customdata,
    hovertemplate=f"{county_col}: %{{customdata[0]}}<br>index: %{{customdata[1]}}<extra></extra>",
))


##### POINT TRACES #####

denver_boundary = prepare_boundary_points("shape_files/denver_uza_akcicek.shp")

                
print(f"denver_boundary.gemetry {denver_boundary.geometry.iloc[0]}")
# all_within_denver = expanded_ct_sensed_gdf[(expanded_ct_sensed_gdf.within(denver_boundary.geometry.iloc[0]))]
colorado_points = expanded_ct_sensed_gdf

# Lists for plotting
start_lon = []
start_lat = []
end_lon = []
end_lat = []
line_lons = []
line_lats = []
start_points_countys = []
end_points_countys = []


for idx, row in colorado_points.iterrows():
    lon_start, lat_start = row['start_loc']['coordinates']
    lon_end, lat_end = row['end_loc']['coordinates']

    start_lon.append(lon_start)
    start_lat.append(lat_start)

    end_lon.append(lon_end)
    end_lat.append(lat_end)

    line_lons.extend([lon_start, lon_end, None])
    line_lats.extend([lat_start, lat_end, None])

    # Determine county for each point - use PREPPED COUNTIES
    start_points_countys.append(find_county_for_point(lon_start, lat_start, prepared_counties))
    end_points_countys.append(find_county_for_point(lon_end, lat_end, prepared_counties))

# create mapping between points and counties
start_customdata = [
    (idx, index_to_county.get(idx, "Unknown"))
    for idx in start_points_countys
]

end_customdata = [
    (idx, index_to_county.get(idx, "Unknown"))
    for idx in end_points_countys
]

# Identify user trips and pattern stats
relationship_stats = {}  
# key: (countyA, countyB)
# value: { "users": set(), "trip_count": int }

for row, start_c, end_c in zip(colorado_points.itertuples(), start_points_countys, end_points_countys):
    if start_c is None or end_c is None:
        continue

    user_id = row.user_id  # replace if different field name

    # normalize pair (bidirectional)
    a, b = sorted([start_c, end_c])
    key = (index_to_county[a], index_to_county[b])

    if key not in relationship_stats:
        relationship_stats[key] = {"users": set(), "trip_count": 0}
        print(f"new entry created for {key}")

    relationship_stats[key]["users"].add(user_id)
    relationship_stats[key]["trip_count"] += 1

from pprint import pprint
print("Relationship Stats:")
pprint(relationship_stats)

# CSV Data
rows = []
for (a, b), v in relationship_stats.items():
    rows.append({
        "county_a_name": a,
        "county_b_name": b,
        "county_a_index": county_index_map[a],
        "county_b_index": county_index_map[b],
        "unique_users": len(v["users"]),
        "trip_count": v["trip_count"],
         "user_ids": [str(u) for u in v["users"]],  # <-- convert UUID → string
        # "user_ids": list(v["users"]), 
    })

relationship_df = pd.DataFrame(rows)
relationship_df.to_csv("county_relationships.csv", index=False)

print("Exported county_relationships.csv")


############

# start center
center_lon = start_lon[0]
center_lat = start_lat[0]


# Draw boundaries
lats = []
lons = []
names = []

# print(geo_df.head())
# print(geo_df.columns

# ADD DENVER BOUNDARY TO PLOT
for geom in denver_boundary.geometry:
    # If it's a single polygon
    if isinstance(geom, Polygon):
        polys = [geom]

    # If it's a multipolygon
    elif isinstance(geom, MultiPolygon):
        polys = list(geom.geoms)

    else:
        continue

    # Extract exterior coordinates for each polygon
    for poly in polys:
        x, y = poly.exterior.xy
        lons.extend(x)
        lats.extend(y)

        lons.append(None)
        lats.append(None)


# Boundary
fig.add_trace(go.Scattermap(
    lon=lons,
    lat=lats,
    mode="lines",
    line=dict(color="red", width=2),
    name="Denver Boundary",
))

# Trip lines
fig.add_trace(go.Scattermap(
    lon=line_lons,
    lat=line_lats,
    mode="lines",
    line=dict(color="gray", width=1),
    opacity=0.4,
    name="Trips",
))

# Plot start points
fig.add_trace(go.Scattermap(
    lon=start_lon,
    lat=start_lat,
    mode="markers",
    marker=dict(size=6, color="blue"),
    name="Start Locations",
    customdata=start_customdata,
    hovertemplate="Start County Index: %{customdata[0]}<br>" +
                  "Start County Name: %{customdata[1]}<extra></extra>",
))

# Plot end points
fig.add_trace(go.Scattermap(
    lon=end_lon,
    lat=end_lat,
    mode="markers",
    marker=dict(size=6, color="green"),
    name="End Locations",
    customdata=end_customdata,
    hovertemplate="End County Index: %{customdata[0]}<br>" +
                  "End County Name: %{customdata[1]}<extra></extra>",
))


# Map center over Colorado
center_lon, center_lat = -105.5844, 39.7392

fig.update_layout(
    legend=dict(yanchor="top", y=0.75, xanchor="right"),
    height=1000,
    margin=dict(l=0, r=0, b=0, t=0),
    autosize=True,
    # MapLibre tile maps use `layout.map`
    map=dict(
        center=dict(lat=center_lat, lon=center_lon),
        zoom=6,
        style="open-street-map"
    )
)

# CLUSTERING K-ANONYMOUSLY


fig.show()
fig.write_html(
    "colorado_census_boundaries_scattermap.html",
    include_plotlyjs="cdn",
    full_html=True
)


# user : (startCounty, endCounty)
# for every county relationship, how many users those trips were from


# CountyA --> CountyB
# (CountyB --> CountyA)
# Bidirectional

# CountyA --> CountyA

# Can we show how many users? Or use a heatmap? If only a couple of users, then we don't show
# We can show trips -- after CONFIRMING K-ANNONYMITY

In [ ]:
# Third try
print("3")

In [ ]:
def polygon_exterior_coords(geom, tolerance=0.0005):

    lons, lats = [], []
    if geom is None:
        return lons, lats

    try:
        geom_simpl = geom.simplify(tolerance, preserve_topology=True)
    except Exception:
        geom_simpl = geom

    # Normalize to list of polygons
    if isinstance(geom_simpl, Polygon):
        polys = [geom_simpl]
    elif isinstance(geom_simpl, MultiPolygon):
        polys = list(geom_simpl.geoms)
    else:
        polys = []
        try:
            for g in geom_simpl.geoms:
                if isinstance(g, Polygon):
                    polys.append(g)
        except Exception:
            pass

    for poly in polys:
        if poly.is_empty or poly.exterior is None:
            continue
        x, y = poly.exterior.xy
        lons.extend(list(x))
        lats.extend(list(y))
        # separator for multi-polygons in the same trace
        lons.append(None)
        lats.append(None)

    return lons, lats

def find_county_for_point(lon, lat, prepared_counties):
    pt = Point(lon, lat)

    for geom_prepared, idx, bounds in prepared_counties:
        minx, miny, maxx, maxy = bounds

        # bounding-box prefilter
        if not (minx <= lon <= maxx and miny <= lat <= maxy):
            continue

        # precise containment test
        if geom_prepared.contains(pt):
            return idx


In [ ]:
# Choropleth + normal
import json
import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import Point
from shapely.prepared import prep
import plotly.graph_objects as go
import plotly.graph_objects as go


# PREPARE BOUNDARIES RETURNS A LIST OF GEODATAFRAMES - PROCESS MULTIPLE SHAPEFILES!!!
census_boundaries = prepare_boundary("colorado_counties_only/")  # list[GeoDataFrame]
# Graph FIRST shapefilefrom array of shape files
# all_gdf = census_boundaries[0]
all_gdf = census_boundaries[0].copy()

# Choroplethmap uses longitude/latitude coordinates
if all_gdf.crs is not None and all_gdf.crs.to_epsg() != 4326:
    all_gdf = all_gdf.to_crs(4326)



##### COUNTY TRACES #####
# Identify county name column
county_col = "COUNTY" if "COUNTY" in all_gdf.columns else None
if county_col is None:
    # Try common alternatives; otherwise, pick first non-geometry string column
    candidates = [c for c in all_gdf.columns if c != "geometry"]
    string_cols = [c for c in candidates if all_gdf[c].astype(str).notna().all()]
    county_col = string_cols[0] if string_cols else None
if county_col is None:
    raise ValueError("Could not find a county name column (e.g., 'COUNTY'). Please provide one.")

# Create a unique county index across all rows
all_gdf["county_index"] = range(len(all_gdf))
# Keep a dict for programmatic use
county_index_map = dict(zip(all_gdf[county_col], all_gdf["county_index"]))


prepared_counties = [
    (prep(row.geometry), row.county_index, row.geometry.bounds)
    for row in all_gdf.itertuples(index=False)
]

# Keep a dict if needed elsewhere
county_index_map = dict(zip(all_gdf[county_col], all_gdf["county_index"]))
index_to_county = dict(zip(all_gdf["county_index"], all_gdf[county_col]))
print(county_index_map)

# ----- Figure -----
fig = go.Figure()
# fig = go.FigureWidget()

county_trace_map = {}
county_geojson = json.loads(
    all_gdf[["county_index", "geometry"]].to_json()
)

# Add choropleth
fig.add_trace(
    go.Choroplethmap(
        geojson=county_geojson,

        locations=all_gdf["county_index"],

        z=all_gdf[color_col],
        
        featureidkey="properties.county_index",

        colorscale="Blues",

        marker=dict(
            opacity=0.30,

            line=dict(
                color="white",
                width=1
            )
        ),

        customdata=all_gdf[
            [county_col, "county_index"]
        ].to_numpy(),

        hovertemplate=(
            f"{county_col}: %{{customdata[0]}}<br>"
            "County Index: %{customdata[1]}<br>"
            "Value: %{z}"
            "<extra></extra>"
        ),

        name="Counties",
    )
)

# 1) Filled polygon per county (visuals)
for row in all_gdf.itertuples(index=False):
    county_name = getattr(row, county_col)
    idx_val = row.county_index

    lons, lats = polygon_exterior_coords(row.geometry)
    if not lons:
        continue
    # Store the index of this polygon trace
    county_trace_map[idx_val] = len(fig.data) - 1

# 2) Interior hover via center markers (one per county)
rep_pts = all_gdf.geometry.representative_point()   # guaranteed inside polygon
points_lon = [p.x for p in rep_pts]
points_lat = [p.y for p in rep_pts]

customdata = pd.DataFrame({
    "county_name": all_gdf[county_col],
    "county_index": all_gdf["county_index"]
}).values

fig.add_trace(go.Scattermap(
    lon=points_lon,
    lat=points_lat,
    mode="markers",
    marker=dict(size=10, color="rgba(0,0,0,1)"),
    name="County centers",
    customdata=customdata,
    hovertemplate=f"{county_col}: %{{customdata[0]}}<br>index: %{{customdata[1]}}<extra></extra>",
))

##### POINT TRACES #####

denver_boundary = prepare_boundary_points("shape_files/denver_uza_akcicek.shp")

                
print(f"denver_boundary.gemetry {denver_boundary.geometry.iloc[0]}")
# all_within_denver = expanded_ct_sensed_gdf[(expanded_ct_sensed_gdf.within(denver_boundary.geometry.iloc[0]))]
colorado_points = expanded_ct_sensed_gdf

# Lists for plotting
start_lon = []
start_lat = []
end_lon = []
end_lat = []
line_lons = []
line_lats = []
start_points_countys = []
end_points_countys = []

# Create links between points and counties
for idx, row in colorado_points.iterrows():
    lon_start, lat_start = row['start_loc']['coordinates']
    lon_end, lat_end = row['end_loc']['coordinates']

    start_lon.append(lon_start)
    start_lat.append(lat_start)

    end_lon.append(lon_end)
    end_lat.append(lat_end)

    line_lons.extend([lon_start, lon_end, None])
    line_lats.extend([lat_start, lat_end, None])

    # Determine county for each point - use PREPPED COUNTIES
    start_points_countys.append(find_county_for_point(lon_start, lat_start, prepared_counties))
    end_points_countys.append(find_county_for_point(lon_end, lat_end, prepared_counties))

# create mapping between points and counties
start_customdata = [
    (idx, index_to_county.get(idx, "Unknown"))
    for idx in start_points_countys
]

end_customdata = [
    (idx, index_to_county.get(idx, "Unknown"))
    for idx in end_points_countys
]

# Identify user trips and pattern stats
relationship_stats = {}  
# key: (countyA, countyB)
# value: { "users": set(), "trip_count": int }

for row, start_c, end_c in zip(colorado_points.itertuples(), start_points_countys, end_points_countys):
    if start_c is None or end_c is None:
        continue

    user_id = row.user_id  # replace if different field name

    # normalize pair (bidirectional)
    a, b = sorted([start_c, end_c])
    key = (index_to_county[a], index_to_county[b])

    if key not in relationship_stats:
        relationship_stats[key] = {"users": set(), "trip_count": 0}
        print(f"case where no trips for trips exist for {key}")

    relationship_stats[key]["users"].add(user_id)
    relationship_stats[key]["trip_count"] += 1

from pprint import pprint
print("Relationship Stats:")
pprint(relationship_stats)

# CSV Data
rows = []
for (a, b), v in relationship_stats.items():
    rows.append({
        "county_a_name": a,
        "county_b_name": b,
        "county_a_index": county_index_map[a],
        "county_b_index": county_index_map[b],
        "unique_users": len(v["users"]),
        "trip_count": v["trip_count"],
         "user_ids": [str(u) for u in v["users"]],  # <-- convert UUID → string
        # "user_ids": list(v["users"]), 
    })

relationship_df = pd.DataFrame(rows)
relationship_df.to_csv("county_relationships.csv", index=False)

print("Exported county_relationships.csv")


############

# start center
center_lon = start_lon[0]
center_lat = start_lat[0]


# Draw boundaries
lats = []
lons = []
names = []

# print(geo_df.head())
# print(geo_df.columns

# ADD DENVER BOUNDARY TO PLOT
for geom in denver_boundary.geometry:
    # If it's a single polygon
    if isinstance(geom, Polygon):
        polys = [geom]

    # If it's a multipolygon
    elif isinstance(geom, MultiPolygon):
        polys = list(geom.geoms)

    else:
        continue

    # Extract exterior coordinates for each polygon
    for poly in polys:
        x, y = poly.exterior.xy
        lons.extend(x)
        lats.extend(y)

        lons.append(None)
        lats.append(None)


# Boundary
fig.add_trace(go.Scattermap(
    lon=lons,
    lat=lats,
    mode="lines",
    line=dict(color="red", width=2),
    name="Denver Boundary",
))

# Trip lines
fig.add_trace(go.Scattermap(
    lon=line_lons,
    lat=line_lats,
    mode="lines",
    line=dict(color="gray", width=1),
    opacity=0.4,
    name="Trips",
))

# Plot start points
fig.add_trace(go.Scattermap(
    lon=start_lon,
    lat=start_lat,
    mode="markers",
    marker=dict(size=6, color="blue"),
    name="Start Locations",
    customdata=start_customdata,
    hovertemplate="Start County Index: %{customdata[0]}<br>" +
                  "Start County Name: %{customdata[1]}<extra></extra>",
))

# Plot end points
fig.add_trace(go.Scattermap(
    lon=end_lon,
    lat=end_lat,
    mode="markers",
    marker=dict(size=6, color="green"),
    name="End Locations",
    customdata=end_customdata,
    hovertemplate="End County Index: %{customdata[0]}<br>" +
                  "End County Name: %{customdata[1]}<extra></extra>",
))


# Map center over Colorado
center_lon, center_lat = -105.5844, 39.7392

fig.update_layout(
    clickmode="event+select",
    height=1000,
    margin=dict(l=0, r=0, b=0, t=0),
    autosize=True,
    legend=dict(yanchor="top",y=0.75,xanchor="right",x=1),
    map=dict(
        center=dict(lat=center_lat, lon=center_lon),
        zoom=6,
        style="open-street-map"
    )
)
# CLUSTERING K-ANONYMOUSLY

fig.show()

fig.write_html(
    "colorado_census_boundaries_scattermap.html",
    include_plotlyjs="cdn",
    full_html=True
)


# user : (startCounty, endCounty)
# for every county relationship, how many users those trips were from


# CountyA --> CountyB
# (CountyB --> CountyA)
# Bidirectional

# CountyA --> CountyA

# Can we show how many users? Or use a heatmap? If only a couple of users, then we don't show
# We can show trips -- after CONFIRMING K-ANNONYMITY